In [1]:
DATASETS = [
    {'path': '/media/alvarinho/dados/Datasets/raw/emea en-es.txt/', 'src_file': 'EMEA.en-es.en', 'tgt_file': 'EMEA.en-es.es'},
    {'path': '/media/alvarinho/dados/Datasets/raw/emea en-pt.txt/', 'src_file': 'EMEA.en-pt.en', 'tgt_file': 'EMEA.en-pt.pt'},
 
    {'path': '/media/alvarinho/dados/Datasets/raw/paracrawl en-es.txt/', 'src_file': 'ParaCrawl.en-es.en', 'tgt_file': 'ParaCrawl.en-es.es'},
    {'path': '/media/alvarinho/dados/Datasets/raw/paracrawl en-pt.txt/', 'src_file': 'ParaCrawl.en-pt.en', 'tgt_file': 'ParaCrawl.en-pt.pt'},
    
    {'path': '/media/alvarinho/dados/Datasets/raw/scielo en-pt.txt/', 'src_file': 'SciELO.en-pt.en', 'tgt_file': 'SciELO.en-pt.pt'},

    {'path': '/media/alvarinho/dados/Datasets/raw/wikimatrix en-es.txt/', 'src_file': 'WikiMatrix.en-es.en', 'tgt_file': 'WikiMatrix.en-es.es'},
    {'path': '/media/alvarinho/dados/Datasets/raw/wikimatrix en-pt.txt/', 'src_file': 'WikiMatrix.en-pt.en', 'tgt_file': 'WikiMatrix.en-pt.pt'},
]

In [2]:
from src.clean_character import CleanCharacter
from src.deduplicator import ExactDuplicator
from src.lang_identify import LangIdentifier
from src.length_clash import LengthClash
from src.numbers_clash import NumbersClash
from src.unidecode_norm import UnidecodeNorm
from typing import List
from src.processor import Processor

In [ ]:
class Pipeline:
    def __init__(self):
        self.steps: List[Processor] = []

    def add_step(self, step):
        self.steps.append(step)

    def process(self, kwargs):
        result_total = []
        metrics = {}
        for step in self.steps:
            step_name = step.__class__.__name__
            # print('Processando step:', )
            (src_line, tgt_line), eval, step_metrics = step.apply_pairs(**kwargs)
            result_total.append(eval)
            metrics[step_name] = step_metrics
            kwargs['text1'] = src_line
            kwargs['text2'] = tgt_line
            
            # print(f"   EVAL: {eval}")
        return src_line, tgt_line, all(result_total), metrics

pipe = Pipeline()
pipe.add_step(CleanCharacter())
pipe.add_step(UnidecodeNorm())
pipe.add_step(LangIdentifier(model_path=None, threshold=0.75, k=1))
pipe.add_step(LengthClash(max_length_diff_ratio=2.0))
pipe.add_step(NumbersClash(threshold=0.75))

In [4]:
dataset_amostra = DATASETS[0]

In [5]:
path = dataset_amostra['path']
src_file = dataset_amostra['src_file']
tgt_file = dataset_amostra['tgt_file']

with open(path + src_file, 'r', encoding='utf-8') as f_src, open(path + tgt_file, 'r', encoding='utf-8') as f_tgt:
    for i, (src_line, tgt_line) in enumerate(zip(f_src, f_tgt)):
        src_line, tgt_line, eval, metrics = pipe.process(
            {'text1': src_line.strip(), 
             'text2': tgt_line.strip(), 
             'expected_lang1': 'en', 
             'expected_lang2': 'es'
             })  

        if(eval):
            print(f"{i+1}: SRC: {src_line.strip()}")
            print(f"    TGT: {tgt_line.strip()}")
            print(f"    EVAL: {eval}")  
            print(f"    METRICS: {metrics}")
            break    

43: SRC: How has Abilify been studied?
    TGT: ¿Qué tipo de estudios se han realizado con Abilify?
    EVAL: True
    METRICS: {'CleanCharacter': {}, 'UnidecodeNorm': {}, 'LangIdentifier': {'prob1': np.float64(0.9997584819793701), 'prob2': np.float64(0.971524715423584)}, 'LengthClash': {'length_diff_ratio': 1.8}, 'NumbersClash': {}}


In [6]:
# contar a quantidade de linhas
with open(path + en_file, 'r', encoding='utf-8') as f_en:
    en_lines = sum(1 for _ in f_en)

print(f"Quantidade de linhas no arquivo {en_file}: {en_lines}")

NameError: name 'en_file' is not defined